| n_chunk_files | total_chunks | n_embedding_files | total_embeddings |
|-------------:|-------------:|------------------:|-----------------:|
| 38189        | 369734       | 38189             | 369734           |


In [0]:
from dbx_hybrid_search import hybrid_search_rrf
import openai
from pyspark.sql import functions as F
from LLMs_call import load_api_key, ask_gpt, build_messages
import markdown

openai.api_key = load_api_key("gpt_api_key", "config.yaml")

In [0]:
def display_MD(md):
    displayHTML(markdown.markdown(md, extensions=["fenced_code", "tables"]))

In [0]:
def RAG_pipe_main(spark, query, LLM_model="gpt-5-mini-2025-08-07", temperature=None,
                  top_each=100, top_n=20, rrf_k=20, return_with_text=True):
    search_df = hybrid_search_rrf(spark, query, top_each=top_each, top_n=top_n, rrf_k=rrf_k, return_with_text=return_with_text)
    search_text_collection = search_df.orderBy(F.col("rrf_score").desc()).select(F.col("text")).limit(10).collect()
    messages = build_messages(text_collection=search_text_collection, user_query=query)
    answer = ask_gpt(
        messages=messages,
        model=LLM_model,
        temperature=temperature,
    )
    return search_df, answer

In [0]:
query = "How did Harris County estimate the number of homes that flooded after Hurricane Harvey, and what data sources did they use?"


In [0]:
search_df, answer = RAG_pipe_main(spark,
                       query,
                       LLM_model="gpt-4o-mini",
                       temperature=0.2)

In [0]:
display(search_df)

In [0]:
display_MD(answer)